# 03. BERTopic

Runs BERTopic over the cleaned posts to find discussion clusters. Frequency analysis tells us which words are common, but not which posts are about the same thing. BERTopic groups posts by meaning instead of by shared vocabulary, so posts describing the same problem in different words end up together.

The output here is the keyword vocabulary we used to query PubMed and ClinicalTrials.gov, not a finished model.

**Run from `notebooks/Reddit_Data/`.** A GPU makes the embedding step much faster.

**Input:** `data/interim/cleaned_submissions.parquet` (from notebook 01)
**Output:** `data/interim/topic_info.csv`

## Prepare the text

In [ ]:
import re
import pandas as pd

df = pd.read_parquet("../../data/interim/cleaned_submissions.parquet")

df = df.copy()
df["selftext"] = df["selftext"].fillna("")

# Second pass in case anything slipped through notebook 01
junk = {"[removed]", "[deleted]", ""}
df["selftext_clean"] = df["selftext"].apply(lambda x: "" if x.strip() in junk else x)

# Titles carry a lot of topical signal on Reddit, so they go in with the body
df["text"] = (df["title"].str.strip() + ". " + df["selftext_clean"].str.strip()).str.strip()

df["text"] = df["text"].apply(lambda x: re.sub(r"http\S+|www\.\S+", "", x))
df["text"] = df["text"].apply(lambda x: re.sub(r"\s+", " ", x).strip())

# Too short to carry a topic
df = df[df["text"].str.len() > 15].reset_index(drop=True)
print(len(df), "posts after cleaning")

## Run BERTopic

`min_topic_size` is the parameter that mattered most. It sets the smallest number of posts a cluster can contain. We settled on 50. We also tried 100, which cut the model from 39 topics to 24 by merging narrow clusters into two large generic ones, and lost exactly the specific topics we were looking for (boric acid, vaginal probiotics, ferritin, thyroid, pelvic floor).

In [ ]:
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer

vectorizer_model = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),   # bigrams so "boric acid" and "night sweats" stay intact
    min_df=5,             # drop rare tokens and misspellings
)

topic_model = BERTopic(
    vectorizer_model=vectorizer_model,
    min_topic_size=50,
    nr_topics="auto",              # merge similar topics automatically
    calculate_probabilities=False, # faster, and we do not need per-doc confidence
    verbose=True,
)

topics, _ = topic_model.fit_transform(df["text"].tolist())
df["topic"] = topics

### Note on reproducibility

BERTopic seeds UMAP randomly, so topic numbering and cluster boundaries shift a little between runs. To pin them, pass a seeded UMAP model:

```python
from umap import UMAP
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0,
                  metric="cosine", random_state=42)
topic_model = BERTopic(umap_model=umap_model, ...)
```

We did not use a seed for the run reported in the paper, so adding one will give a stable model but not an exact match to our 39 topics.

## Inspect the topics

In [ ]:
topic_info = topic_model.get_topic_info()
topic_info.head(20)

In [ ]:
# Topic -1 is the outlier group. A large one is expected here, since many posts
# are broad requests for reassurance or long accounts covering several concerns.
n_outliers = (df["topic"] == -1).sum()
print(f"{len(topic_info) - 1} topics")
print(f"{n_outliers} outlier posts ({n_outliers / len(df):.1%})")

In [ ]:
topic_info.to_csv("../../data/interim/topic_info.csv", index=False)